
# Phase 8R — fully wired from the original Phase-8B experiment

This notebook is reconstructed directly from the original Phase-8B notebook, its raw-results CSV, and its RAG manifest.

It preserves exactly the original:
- six evidence candidates per query;
- `context_for_mask(...)`;
- RAG prompt format;
- answer normalization;
- exact match / token-F1 scoring;
- local model family `Qwen/Qwen3-0.6B`.

## Important methodological point

The original Phase-8B generation used:

```python
do_sample=False
```

so rerunning the same prompt repeatedly would normally give the same answer and **would not provide useful sampling uncertainty**.

Therefore this notebook supports two modes:

```python
REPEAT_MODE = "stochastic"
```

Recommended for uncertainty estimation. All repeated measurements are rerun with the **same stochastic decoding policy**.

or

```python
REPEAT_MODE = "deterministic_check"
```

Useful only to test reproducibility; confidence intervals will generally collapse.

The original deterministic run remains the Phase-8B/10C baseline and is kept separate from the stochastic repeat experiment.


In [1]:

# Optional installation, if needed:
# !pip install -U torch transformers "huggingface-hub<1.0" accelerate safetensors pandas numpy

from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import os
import string
import time
import unicodedata

import numpy as np
import pandas as pd

BASE_SEED = 20260925

# Prefer the original filenames; UUID names are fallbacks for the currently uploaded copies.
def first_existing(*names):
    for name in names:
        if Path(name).exists():
            return name
    raise FileNotFoundError(
        "None of these input files exists:\n" + "\n".join(map(str, names))
    )

MANIFEST_PATH = first_existing(
    "phase8B_rag_manifest.jsonl",
    "f719b2e9-a327-411d-882a-a7c9fa40a6e2.jsonl",
)

BASELINE_CSV = first_existing(
    "phase8B_rag_raw_results.csv",
    "a2182468-b79c-4432-9821-055437001b1f.csv",
)

OUTPUT_CSV = "phase8R_wired_repeated_raw_results.csv"
CHECKPOINT_CSV = "phase8R_wired_checkpoint.csv"

LOCAL_MODEL_ID = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 32
QUALITY_METRIC = "token_f1"

# Recommended: true repeated stochastic measurements.
REPEAT_MODE = "stochastic"  # "stochastic" or "deterministic_check"

# All repeated samples in the new experiment use the same policy.
R_TARGET = 4

# Stochastic-decoding settings.
TEMPERATURE = 0.7
TOP_P = 0.9

# None -> automatically use non-flat baseline queries only.
QUERY_IDS = None

# Leave False until you have inspected the plan.
RUN_MODEL = True

print("manifest:", MANIFEST_PATH)
print("baseline:", BASELINE_CSV)


manifest: phase8B_rag_manifest.jsonl
baseline: phase8B_rag_raw_results.csv


In [2]:

# Load and cross-check the supplied manifest + original raw CSV.
rag_items = [
    json.loads(line)
    for line in open(MANIFEST_PATH, encoding="utf-8")
    if line.strip()
]

baseline = pd.read_csv(BASELINE_CSV)

manifest_sha256 = hashlib.sha256(Path(MANIFEST_PATH).read_bytes()).hexdigest()

if "manifest_sha256" in baseline.columns:
    csv_hashes = set(baseline["manifest_sha256"].dropna().astype(str))
    if csv_hashes != {manifest_sha256}:
        raise RuntimeError(
            f"Manifest/CSV mismatch: manifest={manifest_sha256}, csv={csv_hashes}"
        )

manifest_qids = {str(x["query_id"]) for x in rag_items}
csv_qids = set(baseline["query_id"].astype(str).unique())

if manifest_qids != csv_qids:
    raise RuntimeError("Query IDs in manifest and CSV do not match.")

candidate_counts = {len(x["candidates"]) for x in rag_items}
if len(candidate_counts) != 1:
    raise RuntimeError(f"Inconsistent candidate counts: {candidate_counts}")

N_CANDIDATES = next(iter(candidate_counts))
N_SUBSETS = 1 << N_CANDIDATES
FULL_MASK = N_SUBSETS - 1

print("manifest SHA256:", manifest_sha256)
print("queries:", len(rag_items))
print("candidates/query:", N_CANDIDATES)
print("subsets/query:", N_SUBSETS)
print("baseline rows:", len(baseline))
print("manifest/CSV consistency: OK")


manifest SHA256: c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13
queries: 4
candidates/query: 6
subsets/query: 64
baseline rows: 256
manifest/CSV consistency: OK


In [3]:

# Exact scoring functions from the original notebook.
ARTICLES = {"a", "an", "the"}

def normalize_answer(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    toks = [t for t in s.split() if t not in ARTICLES]
    return " ".join(toks)

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred, gold):
    p = normalize_answer(pred).split()
    g = normalize_answer(gold).split()

    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0

    common = Counter(p) & Counter(g)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return 2 * precision * recall / (precision + recall)

def quality_score(pred, gold):
    if QUALITY_METRIC == "exact_match":
        return exact_match(pred, gold)
    if QUALITY_METRIC == "token_f1":
        return token_f1(pred, gold)
    raise ValueError(QUALITY_METRIC)


In [4]:

# Exact mask/context/prompt reconstruction from the original notebook.
def mask_to_indices(mask):
    return [
        i
        for i in range(N_CANDIDATES)
        if mask & (1 << i)
    ]

def mask_order(mask):
    return int(mask).bit_count()

def context_for_mask(item, mask):
    ids = mask_to_indices(mask)
    selected = [item["candidates"][i] for i in ids]

    if not selected:
        context = "(no retrieved context)"
    else:
        blocks = []
        for i in ids:
            c = item["candidates"][i]
            blocks.append(
                f"[C{i+1}] {c['title']}\n{c['text']}"
            )
        context = "\n\n".join(blocks)

    proxy_tokens = sum(
        item["candidates"][i]["token_proxy"]
        for i in ids
    )

    return selected, context, int(proxy_tokens)

SYSTEM_PROMPT = (
    "Answer the question using only the retrieved context supplied by the user. "
    "If the context is insufficient, answer UNKNOWN. "
    "Return only the shortest answer span; do not explain."
)

def render_rag_prompt(item, mask):
    _, context, _ = context_for_mask(item, mask)

    return (
        f"QUESTION\n{item['question']}\n\n"
        f"RETRIEVED CONTEXT\n{context}\n\n"
        "SHORT ANSWER\n"
    )



## Load the same local model

The original run was deterministic. In stochastic mode this notebook changes **only the decoding policy**, not the model or prompt.


In [5]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    local_tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_MODEL_ID,
        use_fast=True,
    )
    print("fast tokenizer: OK")
except Exception as e:
    print("fast tokenizer failed:", type(e).__name__, str(e)[:160])
    print("falling back to slow tokenizer")
    local_tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_MODEL_ID,
        use_fast=False,
    )

load_kwargs = {
    "torch_dtype": "auto",
    "low_cpu_mem_usage": True,
}

if DEVICE == "cuda":
    load_kwargs["device_map"] = "auto"

local_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_ID,
    **load_kwargs,
)
local_model.eval()

MODEL_DEVICE = next(local_model.parameters()).device

print("model:", LOCAL_MODEL_ID)
print("device:", MODEL_DEVICE)
print("dtype:", next(local_model.parameters()).dtype)


C:\Users\Mauro\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


fast tokenizer: OK
model: Qwen/Qwen3-0.6B
device: cpu
dtype: torch.bfloat16


In [6]:

def apply_chat_template_robust(messages):
    try:
        return local_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return local_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def stable_seed(query_id, mask, repeat):
    material = f"{BASE_SEED}|{query_id}|{mask}|{repeat}".encode()
    return int(hashlib.sha256(material).hexdigest()[:8], 16)

def call_local_rag(user_prompt, seed):
    t0 = time.perf_counter()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    text = apply_chat_template_robust(messages)

    inputs = local_tokenizer(
        text,
        return_tensors="pt",
    ).to(MODEL_DEVICE)

    input_len = int(inputs["input_ids"].shape[1])

    # Reproducible independent sample per (query, mask, repeat).
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    if REPEAT_MODE == "deterministic_check":
        gen_kwargs = {
            "do_sample": False,
        }
    elif REPEAT_MODE == "stochastic":
        gen_kwargs = {
            "do_sample": True,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
        }
    else:
        raise ValueError(REPEAT_MODE)

    with torch.inference_mode():
        outputs = local_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=local_tokenizer.eos_token_id,
            use_cache=True,
            **gen_kwargs,
        )

    generated = outputs[0, input_len:]
    raw_text = local_tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    return {
        "raw_text": raw_text,
        "returned_model": LOCAL_MODEL_ID,
        "input_tokens": input_len,
        "output_tokens": int(generated.numel()),
        "latency_sec": time.perf_counter() - t0,
        "provider_cost_usd": 0.0,
    }



## Select informative baseline queries

We use the original deterministic run only to choose which queries are worth repeating. It is **not mixed into the stochastic repeated dataset**.


In [7]:

diag = (
    baseline
    .groupby("query_id")
    .agg(
        n_masks=("mask", "nunique"),
        min_F=("quality", "min"),
        max_F=("quality", "max"),
        std_F=("quality", lambda x: float(np.std(pd.to_numeric(x, errors="coerce").dropna(), ddof=0))),
    )
    .reset_index()
)

diag["range_F"] = diag["max_F"] - diag["min_F"]
diag["informative"] = diag["std_F"] > 1e-12

display(diag)

if QUERY_IDS is None:
    selected_queries = diag.loc[diag["informative"], "query_id"].astype(str).tolist()
else:
    selected_queries = [str(x) for x in QUERY_IDS]

print("selected:", selected_queries)
print("new calls:", len(selected_queries) * N_SUBSETS * R_TARGET)


,query_id,n_masks,min_F,max_F,std_F,range_F,informative
0,5a855b5c5542997b5ce3ffeb,64,0.0,0.444444,0.174320,0.444444,True
1,5a860aa6554299211dda2a45,64,0.0,1.000000,0.344430,1.000000,True
2,5a90b1a15542990a984936ab,64,0.0,1.000000,0.173993,1.000000,True
3,5abcd77755429965836004ce,64,0.0,0.000000,0.000000,0.000000,False


selected: ['5a855b5c5542997b5ce3ffeb', '5a860aa6554299211dda2a45', '5a90b1a15542990a984936ab']
new calls: 768



### Cost-saving pilot

If the full run is too slow, set for example:

```python
QUERY_IDS = ["5a860aa6554299211dda2a45"]
R_TARGET = 4
```

That is only

\[
64\times4=256
\]

new calls.

Once the bootstrap is stable, expand to the other informative queries and/or \(R=8\).


In [8]:

item_by_id = {str(x["query_id"]): x for x in rag_items}

plan_rows = []
for query_id in selected_queries:
    for mask in range(N_SUBSETS):
        for repeat in range(R_TARGET):
            plan_rows.append({
                "query_id": query_id,
                "mask": mask,
                "repeat": repeat,
            })

plan = pd.DataFrame(plan_rows)

if Path(CHECKPOINT_CSV).exists():
    checkpoint = pd.read_csv(CHECKPOINT_CSV)
    done = set(
        map(
            tuple,
            checkpoint[["query_id", "mask", "repeat"]]
            .assign(query_id=lambda x: x["query_id"].astype(str))
            .itertuples(index=False, name=None)
        )
    )
else:
    checkpoint = pd.DataFrame()
    done = set()

plan["done"] = [
    (str(q), int(m), int(r)) in done
    for q, m, r in plan[["query_id", "mask", "repeat"]].itertuples(index=False, name=None)
]

pending = plan[~plan["done"]].copy()

print("total planned:", len(plan))
print("already done:", int(plan["done"].sum()))
print("pending:", len(pending))
display(pending.head())


total planned: 768
already done: 0
pending: 768


,query_id,mask,repeat,done
0,5a855b5c5542997b5ce3ffeb,0,0,False
1,5a855b5c5542997b5ce3ffeb,0,1,False
2,5a855b5c5542997b5ce3ffeb,0,2,False
3,5a855b5c5542997b5ce3ffeb,0,3,False
4,5a855b5c5542997b5ce3ffeb,1,0,False


In [9]:

def append_checkpoint(row):
    out = pd.DataFrame([row])
    p = Path(CHECKPOINT_CSV)
    out.to_csv(
        p,
        mode="a",
        header=not p.exists(),
        index=False,
    )

if not RUN_MODEL:
    print(
        "RUN_MODEL=False. Inspect the plan first. "
        "Then set RUN_MODEL=True and rerun this cell."
    )
else:
    for i, p in pending.reset_index(drop=True).iterrows():
        query_id = str(p["query_id"])
        mask = int(p["mask"])
        repeat = int(p["repeat"])

        item = item_by_id[query_id]
        prompt = render_rag_prompt(item, mask)
        _, _, context_token_proxy = context_for_mask(item, mask)

        seed = stable_seed(query_id, mask, repeat)

        print(
            f"[{i+1}/{len(pending)}] "
            f"query={query_id} mask={mask:02d} "
            f"order={mask_order(mask)} repeat={repeat} seed={seed}",
            flush=True,
        )

        try:
            result = call_local_rag(prompt, seed)

            em = exact_match(result["raw_text"], item["answer"])
            f1 = token_f1(result["raw_text"], item["answer"])
            quality = em if QUALITY_METRIC == "exact_match" else f1

            row = {
                "experiment": "phase8R_wired",
                "manifest_sha256": manifest_sha256,
                "repeat_mode": REPEAT_MODE,
                "temperature": TEMPERATURE if REPEAT_MODE == "stochastic" else np.nan,
                "top_p": TOP_P if REPEAT_MODE == "stochastic" else np.nan,
                "seed": seed,
                "provider": "local",
                "requested_model": LOCAL_MODEL_ID,
                "returned_model": result["returned_model"],
                "query_id": query_id,
                "dataset_index": item.get("dataset_index"),
                "question": item["question"],
                "gold_answer": item["answer"],
                "mask": mask,
                "order": mask_order(mask),
                "repeat": repeat,
                "raw_text": result["raw_text"],
                "exact_match": em,
                "token_f1": f1,
                "quality": quality,
                "context_token_proxy": context_token_proxy,
                "input_tokens": result["input_tokens"],
                "output_tokens": result["output_tokens"],
                "latency_sec": result["latency_sec"],
                "provider_cost_usd": result["provider_cost_usd"],
                "error": None,
            }

        except Exception as exc:
            row = {
                "experiment": "phase8R_wired",
                "manifest_sha256": manifest_sha256,
                "repeat_mode": REPEAT_MODE,
                "temperature": TEMPERATURE if REPEAT_MODE == "stochastic" else np.nan,
                "top_p": TOP_P if REPEAT_MODE == "stochastic" else np.nan,
                "seed": seed,
                "provider": "local",
                "requested_model": LOCAL_MODEL_ID,
                "returned_model": None,
                "query_id": query_id,
                "dataset_index": item.get("dataset_index"),
                "question": item["question"],
                "gold_answer": item["answer"],
                "mask": mask,
                "order": mask_order(mask),
                "repeat": repeat,
                "raw_text": None,
                "exact_match": np.nan,
                "token_f1": np.nan,
                "quality": np.nan,
                "context_token_proxy": context_token_proxy,
                "input_tokens": np.nan,
                "output_tokens": np.nan,
                "latency_sec": np.nan,
                "provider_cost_usd": 0.0,
                "error": f"{type(exc).__name__}: {exc}",
            }

        append_checkpoint(row)

print("finished / checkpoint preserved")


[1/768] query=5a855b5c5542997b5ce3ffeb mask=00 order=0 repeat=0 seed=3899442476
[2/768] query=5a855b5c5542997b5ce3ffeb mask=00 order=0 repeat=1 seed=4184167401
[3/768] query=5a855b5c5542997b5ce3ffeb mask=00 order=0 repeat=2 seed=1481465206
[4/768] query=5a855b5c5542997b5ce3ffeb mask=00 order=0 repeat=3 seed=3479647269
[5/768] query=5a855b5c5542997b5ce3ffeb mask=01 order=1 repeat=0 seed=3788519517
[6/768] query=5a855b5c5542997b5ce3ffeb mask=01 order=1 repeat=1 seed=3823794612
[7/768] query=5a855b5c5542997b5ce3ffeb mask=01 order=1 repeat=2 seed=1589296056
[8/768] query=5a855b5c5542997b5ce3ffeb mask=01 order=1 repeat=3 seed=3891227037
[9/768] query=5a855b5c5542997b5ce3ffeb mask=02 order=1 repeat=0 seed=3561069765
[10/768] query=5a855b5c5542997b5ce3ffeb mask=02 order=1 repeat=1 seed=2119326092
[11/768] query=5a855b5c5542997b5ce3ffeb mask=02 order=1 repeat=2 seed=3397657250
[12/768] query=5a855b5c5542997b5ce3ffeb mask=02 order=1 repeat=3 seed=4082326299
[13/768] query=5a855b5c5542997b5ce3ff


## Final validation and export

Phase 10D should use only the new repeated experiment—not the old deterministic repeat 0—so all observations share the same decoding policy.


In [10]:

if not Path(CHECKPOINT_CSV).exists():
    print("No checkpoint yet; run the model cell first.")
else:
    repeated = pd.read_csv(CHECKPOINT_CSV)

    # Keep one row per stochastic measurement key.
    repeated = (
        repeated
        .drop_duplicates(["query_id", "mask", "repeat"], keep="last")
        .sort_values(["query_id", "mask", "repeat"])
        .reset_index(drop=True)
    )

    repeated.to_csv(OUTPUT_CSV, index=False)

    coverage = (
        repeated
        .groupby(["query_id", "mask"])
        .agg(
            n_repeats=("repeat", "nunique"),
            n_valid=("quality", lambda x: pd.to_numeric(x, errors="coerce").notna().sum()),
        )
        .reset_index()
    )

    coverage_summary = (
        coverage
        .groupby("query_id")
        .agg(
            n_masks=("mask", "nunique"),
            min_repeats=("n_repeats", "min"),
            max_repeats=("n_repeats", "max"),
            min_valid=("n_valid", "min"),
        )
        .reset_index()
    )

    coverage_summary["ready_for_10D"] = (
        (coverage_summary["n_masks"] == N_SUBSETS)
        & (coverage_summary["min_valid"] >= 2)
    )

    display(coverage_summary)
    print("saved:", OUTPUT_CSV)


,query_id,n_masks,min_repeats,max_repeats,min_valid,ready_for_10D
0,5a855b5c5542997b5ce3ffeb,64,4,4,4,True
1,5a860aa6554299211dda2a45,64,4,4,4,True
2,5a90b1a15542990a984936ab,64,4,4,4,True


saved: phase8R_wired_repeated_raw_results.csv



## What to send back

After the run, send:

- `phase8R_wired_repeated_raw_results.csv`
- optionally this executed notebook

Then Phase 10D can estimate confidence intervals on the interaction spectrum and bit sensitivities.
